RQ3 Analysis: How does participation in a community-engaged course cause 
students to value the experiences of their community partners?
 
Key Variables:
- community_partners_inclusion (Likert 1-5): Student's view on involving partners in decisions
- community_partners_understanding (Likert 1-5): Student's reported understanding of partner perspectives
- do_patners_help (Likert 1-5): Whether student identifies the partner as a resource
- lack_of_interaction_with_partners (Likert 1-5, reverse-coded): Interaction frequency
- Scenario 3: Ethical data use with community org (behavioral measure)
- Scenario 2: Representing community voice (behavioral measure)
 
Composite: partner_value_score = mean of the 3 core items (+ reverse-coded interaction as robustness check)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_csv('/Users/blairwarren/Documents/Stats 141XP/Final Project/Stats141XP-FinalProject/Data/dat_withSentiment.csv')
df.head(3)

,time,survey_date,last_4_digits_uid,last_name,enrolled_course,major,minor,gender,gender_self_described,ethnicity,...,scenario_2_reason_clean,scenario_3_reason_clean,scenario_4_reason_clean,scenario_5_reason_clean,scenario_1_reason_sentiment,scenario_2_reason_sentiment,scenario_3_reason_sentiment,scenario_4_reason_sentiment,scenario_5_reason_sentiment,overall_scenario_sentiment
0,1/29/2026 15:02:43,1/29/2026,532,Whitney,ELTS120XP,Global studies,"Food studies, global health",Man,NaN,White,...,I would be unsure what to say or do,I would take action. Ask questions.,"I’m not sure if I read this correctly, but it ...",I think a lot of barriers come up around findi...,0.0,-0.771393,0.0,0.575386,-0.608252,-0.160852
1,1/29/2026 22:54:03,1/29/2026,7824,Sleeper,ENGCOMP130DX,Public Affairs,"Professional Writing, Environmental Systems & ...",Woman,NaN,Hispanic/Latinx,...,I would ask the instructor because they probab...,I would definitely consult the instructor. Giv...,I would advise my friend to choose Job B. This...,I would say move the workshops to zoom. This w...,0.0,0.000000,0.0,0.696884,0.569326,0.253242
2,1/30/2026 12:26:40,1/30/2026,3402,Owen,CESC 191AX,Political Science,CESC,Woman,NaN,White,...,I would like to get permission first to speak ...,The results don’t have to necessarily be publi...,I would advise my friend that they may feel mo...,In-person meetings are much more valuable ways...,0.0,0.000000,0.0,0.826918,0.765378,0.318459


Construct Composite Scores

In [7]:
rq3_core = ['community_partners_inclusion', 'community_partners_understanding', 'do_patners_help']
df['partner_value'] = df[rq3_core].mean(axis=1)
 
# Also create version with reverse-coded lack_of_interaction
df['lack_interaction_rev'] = 6 - df['lack_of_interaction_with_partners']
rq3_full = rq3_core + ['lack_interaction_rev']
df['partner_value_full'] = df[rq3_full].mean(axis=1)

Colors

In [8]:
PAL = {
    'Asian / Pacific Islander': '#4C72B0',
    'Hispanic/Latinx': '#DD8452',
    'White': '#55A868',
    'Multiple Ethnicity / Other': '#C44E52'
}
ETH_ORDER = ['Asian / Pacific Islander', 'Hispanic/Latinx', 'White', 'Multiple Ethnicity / Other']

# FIGURE 1: Partner Valuation Item-Level Breakdown
Shows the distribution of responses for each of the 3 core items

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
item_labels = {
    'community_partners_inclusion': 'Partners Should Be\nIncluded in Decisions',
    'community_partners_understanding': 'I Understand\nPartner Perspectives',
    'do_patners_help': 'Partners Are\na Valuable Resource'
}
 
colors_likert = ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#4575b4']
likert_labels = ['1 - Strongly\nDisagree', '2', '3 - Neutral', '4', '5 - Strongly\nAgree']
 
for ax, col in zip(axes, rq3_core):
    counts = df[col].dropna().value_counts().sort_index()
    # Ensure all 5 values present
    for v in [1, 2, 3, 4, 5]:
        if v not in counts.index:
            counts[v] = 0
    counts = counts.sort_index()
    
    bars = ax.bar(counts.index, counts.values, color=colors_likert, edgecolor='white', width=0.7)
    ax.set_title(item_labels[col], fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xticklabels(likert_labels, fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add count labels on bars
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2., h + 0.3, f'{int(h)}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
 
axes[0].set_ylabel('Number of Students', fontsize=12)
 
# Add overall means as annotation
for ax, col in zip(axes, rq3_core):
    m = df[col].mean()
    ax.axvline(x=m, color='black', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.text(m, ax.get_ylim()[1]*0.95, f'Mean: {m:.2f}', ha='center', fontsize=9,
            style='italic', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
 
fig.suptitle('RQ3: How Students Value Community Partner Experiences', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rq3_item_distributions.png', dpi=150, bbox_inches='tight')
plt.close()


# FIGURE 2: Partner Value Composite by Ethnicity (with individual points)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
 
eth_data = df.dropna(subset=['ethnicity', 'partner_value'])
 
# Boxplot + swarmplot
sns.boxplot(data=eth_data, y='ethnicity', x='partner_value', order=ETH_ORDER,
            palette=PAL, width=0.5, boxprops=dict(alpha=0.4), 
            fliersize=0, ax=ax, linewidth=1.5)
sns.stripplot(data=eth_data, y='ethnicity', x='partner_value', order=ETH_ORDER,
              palette=PAL, size=8, alpha=0.7, jitter=0.15, ax=ax)
 
# Add mean markers
for i, eth in enumerate(ETH_ORDER):
    m = eth_data[eth_data['ethnicity']==eth]['partner_value'].mean()
    ax.plot(m, i, 'D', color='black', markersize=10, zorder=5)
    ax.annotate(f'{m:.2f}', (m, i), textcoords='offset points', xytext=(15, -5),
                fontsize=10, fontweight='bold')
 
ax.set_xlabel('Partner Valuation Score (1-5 Scale)', fontsize=12)
ax.set_ylabel('')
ax.set_title('Community Partner Valuation by Race/Ethnicity', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(1.5, 5.5)
 
# Add diamond legend
diamond = mpatches.Patch(facecolor='black', label='◆ = Group Mean')
ax.legend(handles=[diamond], loc='lower left', fontsize=10)
 
plt.tight_layout()
plt.savefig('rq3_partner_value_by_ethnicity.png', dpi=150, bbox_inches='tight')
plt.close()


# FIGURE 3: Partner Value by Belonging Rate & First-Gen Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
# A: By Belonging Rate
grp_belong = df.groupby('belonging_rate')['partner_value'].agg(['mean','std','count'])
grp_belong = grp_belong.reindex(['About the same', 'Higher'])
bars = axes[0].bar(range(len(grp_belong)), grp_belong['mean'], 
                   yerr=grp_belong['std']/np.sqrt(grp_belong['count']),
                   color=['#91bfdb', '#4575b4'], edgecolor='white', width=0.5,
                   capsize=5, error_kw={'linewidth': 1.5})
axes[0].set_xticks(range(len(grp_belong)))
axes[0].set_xticklabels(['About the Same', 'Higher'], fontsize=11)
axes[0].set_ylabel('Mean Partner Value Score', fontsize=11)
axes[0].set_title('Partner Valuation by\nSense of Belonging', fontsize=13, fontweight='bold')
axes[0].set_ylim(3.0, 5.2)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
 
for bar, row in zip(bars, grp_belong.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15,
                 f'{row.mean:.2f}\n(n={int(row.count)})', ha='center', fontsize=10, fontweight='bold')
 
# B: By First-Gen x Transfer
cats = [
    ('Non-First-Gen,\nNon-Transfer', (df['first_gen_college']=='No') & (df['transfer_student']=='No')),
    ('Non-First-Gen,\nTransfer', (df['first_gen_college']=='No') & (df['transfer_student']=='Yes')),
    ('First-Gen,\nNon-Transfer', (df['first_gen_college']=='Yes') & (df['transfer_student']=='No')),
    ('First-Gen,\nTransfer', (df['first_gen_college']=='Yes') & (df['transfer_student']=='Yes')),
]
 
means, ses, ns = [], [], []
for label, mask in cats:
    vals = df.loc[mask, 'partner_value'].dropna()
    means.append(vals.mean())
    ses.append(vals.std() / np.sqrt(len(vals)) if len(vals) > 1 else 0)
    ns.append(len(vals))
 
bar_colors = ['#a6cee3', '#1f78b4', '#fb9a99', '#e31a1c']
bars2 = axes[1].bar(range(4), means, yerr=ses, color=bar_colors, edgecolor='white',
                    width=0.6, capsize=5, error_kw={'linewidth': 1.5})
axes[1].set_xticks(range(4))
axes[1].set_xticklabels([c[0] for c in cats], fontsize=9)
axes[1].set_ylabel('Mean Partner Value Score', fontsize=11)
axes[1].set_title('Partner Valuation by\nFirst-Gen × Transfer Status', fontsize=13, fontweight='bold')
axes[1].set_ylim(3.0, 5.2)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
 
for bar, m, n in zip(bars2, means, ns):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15,
                 f'{m:.2f}\n(n={n})', ha='center', fontsize=9, fontweight='bold')
 
plt.tight_layout()
plt.savefig('rq3_partner_value_belonging_firstgen.png', dpi=150, bbox_inches='tight')
plt.close()


# FIGURE 4: Scenario 3 Analysis (Ethical Data Use - Behavioral RQ3 Measure)
This scenario directly measures how students value community partner autonomy and data sovereignty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
 
# Short labels for scenario choices
s3_map = {
    'I would suggest discussing the ethical concerns with the instructor or organization before moving forward.': 'Discuss Ethics\nw/ Instructor',
    'Given the circumstances, I would agree it is better not to pursue the use of the data.': 'Respect Org\'s\nData Decision',
    'I would take some form of action related to how the data are used.': 'Take Direct\nAction'
}
df['scenario_3_short'] = df['scenario_3'].map(s3_map)
 
# A: Overall distribution with ethnicity breakdown
s3_order = ['Discuss Ethics\nw/ Instructor', "Respect Org's\nData Decision", 'Take Direct\nAction']
ct = pd.crosstab(df['scenario_3_short'], df['ethnicity'])
ct = ct.reindex(s3_order).fillna(0)
 
ct[ETH_ORDER].plot(kind='barh', stacked=True, ax=axes[0], 
                    color=[PAL[e] for e in ETH_ORDER], edgecolor='white', width=0.6)
axes[0].set_xlabel('Number of Students', fontsize=11)
axes[0].set_ylabel('')
axes[0].set_title('Scenario 3: Community Data Ethics\nResponse by Ethnicity', fontsize=13, fontweight='bold')
axes[0].legend(title='Ethnicity', fontsize=8, title_fontsize=9, loc='lower right')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
 
# B: Sentiment of reasoning by choice
s3_sent = df.dropna(subset=['scenario_3_short', 'scenario_3_reason_sentiment'])
sns.boxplot(data=s3_sent, y='scenario_3_short', x='scenario_3_reason_sentiment',
            order=s3_order, palette=['#4575b4', '#91bfdb', '#fc8d59'],
            width=0.5, fliersize=0, ax=axes[1], boxprops=dict(alpha=0.4))
sns.stripplot(data=s3_sent, y='scenario_3_short', x='scenario_3_reason_sentiment',
              order=s3_order, palette=['#4575b4', '#91bfdb', '#fc8d59'],
              size=7, alpha=0.7, jitter=0.1, ax=axes[1])
axes[1].axvline(x=0, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Sentiment Score (Negative ← → Positive)', fontsize=11)
axes[1].set_ylabel('')
axes[1].set_title('Sentiment of Written Reasoning\nby Scenario 3 Choice', fontsize=13, fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
 
plt.tight_layout()
plt.savefig('rq3_scenario3_analysis.png', dpi=150, bbox_inches='tight')
plt.close()


# FIGURE 5: Partner Value vs Prior XP Courses (Dosage Effect)
Does more exposure to community-engaged courses increase partner valuation?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
 
xp_order = ['1', '2', 'More than 2']
xp_colors = ['#fee08b', '#91bfdb', '#4575b4']
 
for i, (xp, color) in enumerate(zip(xp_order, xp_colors)):
    vals = df[df['xp_courses']==xp]['partner_value'].dropna()
    # Jittered strip
    jitter = np.random.normal(i, 0.08, len(vals))
    ax.scatter(jitter, vals, c=color, s=60, alpha=0.6, edgecolors='white', linewidth=0.5, zorder=3)
    # Mean + CI
    m = vals.mean()
    se = vals.std() / np.sqrt(len(vals))
    ax.plot([i-0.2, i+0.2], [m, m], color='black', linewidth=2.5, zorder=4)
    ax.errorbar(i, m, yerr=1.96*se, color='black', linewidth=1.5, capsize=6, zorder=4)
    ax.text(i, m + 0.25, f'{m:.2f}\n(n={len(vals)})', ha='center', fontsize=10, fontweight='bold')
 
ax.set_xticks(range(3))
ax.set_xticklabels(['1 Course', '2 Courses', 'More than 2'], fontsize=11)
ax.set_ylabel('Partner Valuation Score', fontsize=12)
ax.set_title('Partner Valuation by Number of\nPrior Community-Engaged Courses', fontsize=14, fontweight='bold')
ax.set_ylim(1.5, 5.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.axhline(y=df['partner_value'].mean(), color='gray', linestyle='--', alpha=0.3, label='Overall Mean')
ax.legend(fontsize=9)
 
plt.tight_layout()
plt.savefig('rq3_partner_value_xp_dosage.png', dpi=150, bbox_inches='tight')
plt.close()


# STATISTICAL TESTS SUMMARY

In [14]:
print("\n" + "="*60)
print("STATISTICAL TESTS FOR RQ3")
print("="*60)
 
# 1. Mann-Whitney: Partner value by belonging
higher = df[df['belonging_rate']=='Higher']['partner_value'].dropna()
same = df[df['belonging_rate']=='About the same']['partner_value'].dropna()
u, p = stats.mannwhitneyu(higher, same, alternative='two-sided')
print(f"\n1. Mann-Whitney U (Belonging: Higher vs Same)")
print(f"   U = {u:.1f}, p = {p:.4f}")
print(f"   Higher: M={higher.mean():.3f}, SD={higher.std():.3f}")
print(f"   Same:   M={same.mean():.3f}, SD={same.std():.3f}")
print(f"   → {'Significant' if p < 0.05 else 'Not significant'} at α=0.05")
print(f"   → Direction: Students with higher belonging report {'higher' if higher.mean() > same.mean() else 'lower'} partner valuation")
 
# 2. Kruskal-Wallis: Partner value by ethnicity
groups_eth = [g['partner_value'].dropna().values for _, g in df.groupby('ethnicity')]
h, p_kw = stats.kruskal(*groups_eth)
print(f"\n2. Kruskal-Wallis (Partner Value by Ethnicity)")
print(f"   H = {h:.3f}, p = {p_kw:.4f}")
print(f"   → {'Significant' if p_kw < 0.05 else 'Not significant'} at α=0.05")
print(f"   → Ethnicity groups show similar partner valuation levels")
 
# 3. Kruskal-Wallis: Partner value by XP courses
groups_xp = [g['partner_value'].dropna().values for _, g in df.groupby('xp_courses')]
h_xp, p_xp = stats.kruskal(*groups_xp)
print(f"\n3. Kruskal-Wallis (Partner Value by # XP Courses)")
print(f"   H = {h_xp:.3f}, p = {p_xp:.4f}")
print(f"   → {'Significant' if p_xp < 0.05 else 'Not significant'} at α=0.05")
 
# 4. Spearman correlation: partner_value vs overall scenario sentiment
r_sp, p_sp = stats.spearmanr(df['partner_value'].dropna(), 
                               df.loc[df['partner_value'].notna(), 'overall_scenario_sentiment'])
print(f"\n4. Spearman Correlation (Partner Value vs Overall Scenario Sentiment)")
print(f"   rₛ = {r_sp:.3f}, p = {p_sp:.4f}")
print(f"   → {'Significant' if p_sp < 0.05 else 'Not significant'} positive association")
 
# 5. Chi-square: Scenario 3 choice by ethnicity
ct_chi = pd.crosstab(df['ethnicity'], df['scenario_3'])
# Only if cells large enough
print(f"\n5. Scenario 3 Choice × Ethnicity")
print(f"   Note: Expected cell counts too small for valid χ² (n=45)")
print(f"   Descriptive pattern: White students more likely to 'Respect Org's Decision' (43%)")
print(f"   Asian/PI students more spread across choices including 'Take Action' (17%)")
print(f"   Hispanic/Latinx students favor 'Discuss Ethics' (71%) over passive options")
 
# 6. Effect size for belonging comparison
def cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    pooled_std = np.sqrt(((n1-1)*g1.std()**2 + (n2-1)*g2.std()**2) / (n1+n2-2))
    return (g1.mean() - g2.mean()) / pooled_std if pooled_std > 0 else 0
 
d = cohens_d(higher, same)
print(f"\n6. Effect Size (Cohen's d): Belonging → Partner Value")
print(f"   d = {d:.3f}")
size_label = 'small' if abs(d) < 0.5 else ('medium' if abs(d) < 0.8 else 'large')
print(f"   → {size_label.capitalize()} effect size")
 
print("\n" + "="*60)
print("KEY FINDINGS SUMMARY FOR RQ3 SLIDES")
print("="*60)
print("""
1. OVERALL: Students report strong valuation of community partners
   (composite M=4.33, SD=0.71 on 1-5 scale). Understanding of partner 
   perspectives scores highest (M=4.53), while inclusion in decision-making 
   is slightly lower (M=4.21).
 
2. BELONGING MATTERS: Students who report higher belonging also value 
   partners more (M=4.46 vs 4.07), with a moderate effect size (d={:.2f}), 
   though not statistically significant (p=.21) due to small sample.
 
3. ETHNICITY: No significant differences across ethnic groups (p=.84). 
   All groups score high. Hispanic/Latinx students show the most 
   proactive behavioral stance in Scenario 3 (71% choose to discuss ethics).
 
4. DOSAGE: Students with exactly 1 prior XP course show slightly higher 
   partner valuation (M=4.47) than those with 2 (M=4.00), suggesting 
   a possible novelty/enthusiasm effect rather than linear dosage.
 
5. BEHAVIORAL ALIGNMENT: Scenario 3 reveals students overwhelmingly 
   respect community partner autonomy (58% discuss ethics, 36% defer to 
   org), reflecting strong behavioral alignment with attitudinal measures.
 
6. SENTIMENT: Students who wrote about discussing ethical concerns showed 
   near-neutral sentiment (measured, deliberative tone), while those who 
   deferred to the org expressed more negative sentiment (concern, caution).
""".format(d))


STATISTICAL TESTS FOR RQ3

1. Mann-Whitney U (Belonging: Higher vs Same)
   U = 276.0, p = 0.2139
   Higher: M=4.456, SD=0.536
   Same:   M=4.067, SD=0.936
   → Not significant at α=0.05
   → Direction: Students with higher belonging report higher partner valuation

2. Kruskal-Wallis (Partner Value by Ethnicity)
   H = 0.835, p = 0.8410
   → Not significant at α=0.05
   → Ethnicity groups show similar partner valuation levels

3. Kruskal-Wallis (Partner Value by # XP Courses)
   H = 3.423, p = 0.1806
   → Not significant at α=0.05

4. Spearman Correlation (Partner Value vs Overall Scenario Sentiment)
   rₛ = 0.130, p = 0.3953
   → Not significant positive association

5. Scenario 3 Choice × Ethnicity
   Note: Expected cell counts too small for valid χ² (n=45)
   Descriptive pattern: White students more likely to 'Respect Org's Decision' (43%)
   Asian/PI students more spread across choices including 'Take Action' (17%)
   Hispanic/Latinx students favor 'Discuss Ethics' (71%) over pass